[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aqib-ali-29184/pdc-feature-selection/blob/milestone-2/milestone2_member1_addition_operator.ipynb)

In [ ]:
import numpy as np
import pandas as pd
from sklearn.datasets import fetch_openml, make_classification
from sklearn.naive_bayes import GaussianNB
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import accuracy_score
from sklearn.feature_selection import mutual_info_classif
import matplotlib.pyplot as plt
import time
import warnings
warnings.filterwarnings('ignore')

print("✅ All libraries imported successfully!")

Milestone 1 Building Blocks (re-used as-is)


In [ ]:
# ─────────────────────────────────────────────
# DATASET LOADER (Member 1 — Milestone 1)
# ─────────────────────────────────────────────
class DatasetLoader:
    def __init__(self, dataset_name: str):
        self.dataset_name = dataset_name
        self.X = None
        self.y = None

    def load(self):
        print(f"\n📦 Loading dataset: {self.dataset_name} ...")

        if self.dataset_name == "mnist":
            from keras.datasets import mnist
            (X_train, y_train), (X_test, y_test) = mnist.load_data()
            X_all = np.concatenate([X_train, X_test], axis=0)
            y_all = np.concatenate([y_train, y_test], axis=0)
            self.X = X_all.reshape(X_all.shape[0], -1).astype(np.float32)
            self.y = y_all.astype(int)

        elif self.dataset_name == "cifar10_gray":
            from keras.datasets import cifar10
            import cv2
            (X_train, y_train), (X_test, y_test) = cifar10.load_data()
            X_all = np.concatenate([X_train, X_test], axis=0)
            y_all = np.concatenate([y_train.flatten(), y_test.flatten()], axis=0)
            self.X = np.array([
                cv2.cvtColor(img, cv2.COLOR_RGB2GRAY).flatten()
                for img in X_all
            ], dtype=np.float32)
            self.y = y_all

        elif self.dataset_name == "madelon":
            X, y = make_classification(
                n_samples=2600, n_features=500,
                n_informative=20, n_redundant=480,
                n_classes=2, random_state=42
            )
            self.X = X.astype(np.float32)
            self.y = y.astype(int)

        else:
            raise ValueError(f"Unknown dataset: {self.dataset_name}")

        print(f"   Shape     : {self.X.shape}")
        print(f"   Classes   : {np.unique(self.y)}")
        print(f"   Features  : {self.X.shape[1]}")
        return self

    def preprocess(self, sample_size=None):
        if sample_size and sample_size < len(self.X):
            np.random.seed(42)
            idx = np.random.choice(len(self.X), sample_size, replace=False)
            self.X = self.X[idx]
            self.y = self.y[idx]
            print(f"   Subsampled to {sample_size} samples")
        scaler = MinMaxScaler()
        self.X = scaler.fit_transform(self.X)
        print(f"   ✅ Preprocessing done. Feature range: [{self.X.min():.2f}, {self.X.max():.2f}]")
        return self

    def split(self, test_size=0.2, random_state=42):
        X_train, X_test, y_train, y_test = train_test_split(
            self.X, self.y,
            test_size=test_size, random_state=random_state, stratify=self.y
        )
        print(f"   Train: {X_train.shape}, Test: {X_test.shape}")
        return X_train, X_test, y_train, y_test


# ─────────────────────────────────────────────
# NAIVE BAYES EVALUATOR (Member 1 — Milestone 1)
# ─────────────────────────────────────────────
class NaiveBayesEvaluator:
    def __init__(self):
        self.model = GaussianNB()

    def evaluate(self, X_train, X_test, y_train, y_test, feature_mask=None):
        if feature_mask is not None:
            selected = np.where(np.array(feature_mask) == 1)[0]
            if len(selected) == 0:
                return 0.0, 0.0
            X_tr = X_train[:, selected]
            X_te = X_test[:, selected]
        else:
            X_tr, X_te = X_train, X_test

        start = time.time()
        self.model.fit(X_tr, y_train)
        preds = self.model.predict(X_te)
        elapsed = time.time() - start
        acc = accuracy_score(y_test, preds)
        return acc, elapsed


# ─────────────────────────────────────────────
# BINARY MASK ENCODER (Member 2 — Milestone 1)
# ─────────────────────────────────────────────
class BinaryMaskEncoder:
    def __init__(self, n_features: int):
        self.n_features = n_features

    def all_ones(self):
        return np.ones(self.n_features, dtype=int)

    def all_zeros(self):
        return np.zeros(self.n_features, dtype=int)

    def random_mask(self, density=0.5, seed=None):
        if seed is not None:
            np.random.seed(seed)
        return (np.random.rand(self.n_features) < density).astype(int)

    def flip_bit(self, mask, index):
        new_mask = mask.copy()
        new_mask[index] = 1 - new_mask[index]
        return new_mask

    def count_active(self, mask):
        return int(np.sum(mask))

    def get_active_indices(self, mask):
        return np.where(np.array(mask) == 1)[0]

    def mask_to_string(self, mask):
        return ''.join(map(str, mask))


# ─────────────────────────────────────────────
# FITNESS FUNCTION (Member 2 — Milestone 1)
# ─────────────────────────────────────────────
class FitnessFunction:
    def __init__(self, evaluator: NaiveBayesEvaluator, alpha=0.01):
        self.evaluator = evaluator
        self.alpha = alpha

    def evaluate(self, mask, X_train, X_test, y_train, y_test):
        mask = np.array(mask)
        n_total    = len(mask)
        n_selected = int(np.sum(mask))
        if n_selected == 0:
            return 0.0, 0.0, 0, 0.0
        accuracy, time_taken = self.evaluator.evaluate(
            X_train, X_test, y_train, y_test, feature_mask=mask
        )
        feature_ratio = n_selected / n_total
        score = accuracy - (self.alpha * feature_ratio)
        return score, accuracy, n_selected, time_taken


# ─────────────────────────────────────────────
# RESULTS LOGGER (Member 2 — Milestone 1)
# ─────────────────────────────────────────────
class ResultsLogger:
    def __init__(self, dataset_name: str):
        self.dataset_name = dataset_name
        self.records = []

    def log(self, mask, score, accuracy, n_selected, time_taken, label=""):
        encoder = BinaryMaskEncoder(len(mask))
        self.records.append({
            "dataset"    : self.dataset_name,
            "label"      : label,
            "score"      : round(score, 6),
            "accuracy"   : round(accuracy, 6),
            "n_selected" : n_selected,
            "n_total"    : len(mask),
            "feature_%"  : round(100 * n_selected / len(mask), 2),
            "time_s"     : round(time_taken, 6),
            "mask_str"   : encoder.mask_to_string(mask)
        })

    def get_best(self):
        if not self.records:
            return None
        return max(self.records, key=lambda r: r["score"])

    def to_dataframe(self):
        return pd.DataFrame(self.records)

    def save_csv(self, filename=None):
        if filename is None:
            filename = f"fitness_results_{self.dataset_name}.csv"
        df = self.to_dataframe()
        df.to_csv(filename, index=False)
        print(f"✅ Saved {len(df)} results to '{filename}'")
        return filename

    def print_summary(self):
        df = self.to_dataframe()
        if df.empty:
            print("No results logged yet.")
            return
        print(f"\n📋 RESULTS SUMMARY — {self.dataset_name.upper()}")
        print(f"   Total evaluations : {len(df)}")
        print(f"   Best accuracy     : {df['accuracy'].max()*100:.2f}%")
        print(f"   Best score        : {df['score'].max():.6f}")
        print(f"   Avg features used : {df['n_selected'].mean():.1f} / {df['n_total'].iloc[0]}")
        print(f"\n   Top 5 results:")
        top5 = df.nlargest(5, 'score')[['label','accuracy','n_selected','feature_%','score','time_s']]
        print(top5.to_string(index=False))

print("✅ All Milestone 1 classes loaded successfully!")

Information-Gain Feature Bias

In [ ]:
class FeatureBiasComputer:
    """
    Computes an information-gain (IG) based probability distribution
    over features to guide the Addition Operator.

    The distribution is used to bias which zero-bit the operator
    flips to 1, favouring features that are more predictive of the label.

    Parameters
    ----------
    bias_temperature : float
        Controls sharpness of the softmax over IG scores.
        Higher  → more greedy (always picks top-IG feature).
        Lower   → more uniform (more exploration).
        Default 2.0 is a good starting balance.
    random_state : int
        Seed for reproducibility when computing mutual information.
    """

    def __init__(self, bias_temperature: float = 2.0, random_state: int = 42):
        self.bias_temperature = bias_temperature
        self.random_state     = random_state
        self.ig_scores_       = None   # raw IG per feature (set after fit)
        self.probabilities_   = None   # softmax-normalised distribution

    # ──────────────────────────────────────────────────────────
    def fit(self, X_train: np.ndarray, y_train: np.ndarray) -> "FeatureBiasComputer":
        """
        Compute mutual information between each feature and the target label.
        Stores raw IG scores and a softmax-normalised probability vector.

        Parameters
        ----------
        X_train : shape (n_samples, n_features)
        y_train : shape (n_samples,)

        Returns
        -------
        self  (for method chaining)
        """
        print("   ⏳ Computing information-gain scores …", end=" ")
        start = time.time()

        # mutual_info_classif returns IG ≥ 0 for every feature
        ig = mutual_info_classif(
            X_train, y_train,
            discrete_features=False,
            random_state=self.random_state
        )
        self.ig_scores_ = ig

        # Softmax with temperature so the distribution sums to 1
        # and is always well-defined (even if all IG values are equal)
        shifted = ig - ig.max()            # numerical stability
        exp_vals = np.exp(self.bias_temperature * shifted)
        self.probabilities_ = exp_vals / exp_vals.sum()

        elapsed = time.time() - start
        print(f"done ({elapsed:.1f}s)")
        print(f"   Top-5 IG features: indices {np.argsort(ig)[::-1][:5]}")
        print(f"   IG range          : [{ig.min():.4f}, {ig.max():.4f}]")
        return self

    # ──────────────────────────────────────────────────────────
    def get_addition_probabilities(self, mask: np.ndarray) -> np.ndarray:
        """
        Returns a probability vector restricted to *inactive* features (mask == 0).
        Active features (mask == 1) are set to 0 and the remainder
        is re-normalised so the operator only considers adding a new feature.

        Parameters
        ----------
        mask : binary array of length n_features

        Returns
        -------
        probs : normalised probability vector over inactive features
        """
        if self.probabilities_ is None:
            raise RuntimeError("Call .fit() before .get_addition_probabilities()")

        probs = self.probabilities_.copy()
        probs[mask == 1] = 0.0          # zero out already-active features

        total = probs.sum()
        if total == 0:
            # All features already active — return uniform over all
            probs = np.ones(len(mask)) / len(mask)
        else:
            probs /= total

        return probs

    # ──────────────────────────────────────────────────────────
    def top_k_inactive(self, mask: np.ndarray, k: int = 10) -> np.ndarray:
        """
        Returns the indices of the top-k highest-IG inactive features.
        Useful for initialising populations in the parallel framework.

        Parameters
        ----------
        mask : binary array
        k    : number of candidates to return

        Returns
        -------
        indices : np.ndarray of shape (min(k, n_inactive),)
        """
        inactive = np.where(mask == 0)[0]
        if len(inactive) == 0:
            return np.array([], dtype=int)
        ig_inactive = self.ig_scores_[inactive]
        sorted_idx  = np.argsort(ig_inactive)[::-1]
        return inactive[sorted_idx[:k]]

print("✅ FeatureBiasComputer class defined.")

Addition Operator

In [ ]:
class AdditionOperator:
    """
    Milestone 2 — Member 1: Addition Operator

    Responsibilities
    ----------------
    1. Accept a **population** of binary masks.
    2. For each mask, propose a *candidate* by flipping 0→1 only
       (never removing features that are already active).
    3. Use an information-gain bias (FeatureBiasComputer) to prefer
       activating highly predictive features.
    4. Accept the candidate if it has a better fitness score (greedy).
    5. Return the updated population + log every evaluation.

    Interface note (for Hasan's parallel framework)
    ------------------------------------------------
    - Input  : list of np.ndarray masks  →  operate()
    - Output : list of np.ndarray masks  (same length, same indices)
    This makes it trivial to run AdditionOperator and RemovalOperator
    on the *same* shared population list without race conditions,
    because each operator owns its result before the coordinator merges.

    Parameters
    ----------
    fitness_fn        : FitnessFunction (from Milestone 1)
    bias_computer     : FeatureBiasComputer (computed once per dataset)
    n_additions       : int   — how many 0→1 flips to attempt per mask
    use_bias          : bool  — if False, pick inactive features uniformly
                                (useful for ablation comparison)
    random_state      : int
    """

    def __init__(
        self,
        fitness_fn:    FitnessFunction,
        bias_computer: FeatureBiasComputer,
        n_additions:   int  = 3,
        use_bias:      bool = True,
        random_state:  int  = 42,
    ):
        self.fitness_fn    = fitness_fn
        self.bias_computer = bias_computer
        self.n_additions   = n_additions
        self.use_bias      = use_bias
        self.random_state  = random_state
        self._rng          = np.random.default_rng(random_state)

    # ── Internals ─────────────────────────────────────────────

    def _pick_features_to_add(self, mask: np.ndarray) -> np.ndarray:
        """
        Choose which inactive features to activate.

        If use_bias is True  → sample from IG-weighted distribution.
        If use_bias is False → sample uniformly from inactive positions.

        Returns
        -------
        indices : array of positions to flip 0→1
                  (length = min(n_additions, n_inactive))
        """
        inactive = np.where(mask == 0)[0]

        if len(inactive) == 0:
            return np.array([], dtype=int)   # mask is already all-ones

        n_to_add = min(self.n_additions, len(inactive))

        if self.use_bias:
            # Probability vector over ALL features, zeroed for active ones
            probs = self.bias_computer.get_addition_probabilities(mask)
            inactive_probs = probs[inactive]

            # Normalise again in case of floating-point drift
            s = inactive_probs.sum()
            if s == 0:
                inactive_probs = np.ones(len(inactive)) / len(inactive)
            else:
                inactive_probs = inactive_probs / s

            chosen = self._rng.choice(
                inactive, size=n_to_add, replace=False, p=inactive_probs
            )
        else:
            # Uniform random selection (ablation / baseline mode)
            chosen = self._rng.choice(inactive, size=n_to_add, replace=False)

        return chosen

    def _add_features(self, mask: np.ndarray, indices: np.ndarray) -> np.ndarray:
        """
        Flip the given indices from 0 → 1.
        Never touches indices that are already 1.
        Returns a NEW array (original mask is not modified).
        """
        new_mask = mask.copy()
        for idx in indices:
            if new_mask[idx] == 0:          # safety check: only add, never remove
                new_mask[idx] = 1
        return new_mask

    # ── Public API ────────────────────────────────────────────

    def operate(
        self,
        population:  list,
        X_train:     np.ndarray,
        X_test:      np.ndarray,
        y_train:     np.ndarray,
        y_test:      np.ndarray,
        logger:      ResultsLogger = None,
        generation:  int = 0,
    ) -> list:
        """
        Apply the Addition Operator to every mask in the population.

        For each mask:
          1. Propose a candidate by activating n_additions features.
          2. Evaluate candidate fitness.
          3. Keep candidate if it improves fitness (greedy accept).
          4. Log the result.

        Parameters
        ----------
        population  : list of np.ndarray binary masks (current generation)
        X_train, X_test, y_train, y_test : dataset splits
        logger      : ResultsLogger instance (optional — pass one in for full logging)
        generation  : int, used for log labels (e.g. "add_gen3_mask0")

        Returns
        -------
        new_population : list of np.ndarray masks (same length as input)
        stats          : dict with summary statistics for this generation
        """
        new_population = []
        n_improved     = 0
        all_scores     = []

        for i, mask in enumerate(population):
            mask = np.array(mask, dtype=int)

            # ── Evaluate current mask ──────────────────────────
            curr_score, curr_acc, curr_n, curr_t = self.fitness_fn.evaluate(
                mask, X_train, X_test, y_train, y_test
            )

            # ── Propose candidate (add features) ──────────────
            to_add    = self._pick_features_to_add(mask)
            candidate = self._add_features(mask, to_add)

            if len(to_add) == 0:
                # Nothing to add — mask is already fully active
                new_population.append(mask)
                all_scores.append(curr_score)
                if logger:
                    logger.log(
                        mask, curr_score, curr_acc, curr_n, curr_t,
                        label=f"add_gen{generation}_mask{i}_no_inactive"
                    )
                continue

            # ── Evaluate candidate ─────────────────────────────
            cand_score, cand_acc, cand_n, cand_t = self.fitness_fn.evaluate(
                candidate, X_train, X_test, y_train, y_test
            )

            # ── Greedy accept ──────────────────────────────────
            if cand_score > curr_score:
                accepted = candidate
                accepted_score = cand_score
                n_improved += 1
                label = f"add_gen{generation}_mask{i}_improved"
            else:
                accepted = mask
                accepted_score = curr_score
                label = f"add_gen{generation}_mask{i}_kept"

            new_population.append(accepted)
            all_scores.append(accepted_score)

            # ── Log ───────────────────────────────────────────
            if logger:
                # Log the candidate (what was tried)
                logger.log(
                    candidate, cand_score, cand_acc, cand_n, cand_t,
                    label=label
                )

        # ── Generation summary ────────────────────────────────
        stats = {
            "generation"       : generation,
            "population_size"  : len(population),
            "n_improved"       : n_improved,
            "improvement_rate" : n_improved / max(len(population), 1),
            "avg_score"        : float(np.mean(all_scores)) if all_scores else 0.0,
            "best_score"       : float(np.max(all_scores))  if all_scores else 0.0,
        }

        return new_population, stats

    # ── Convenience: single-mask operation (for testing) ──────

    def step(
        self,
        mask:    np.ndarray,
        X_train: np.ndarray,
        X_test:  np.ndarray,
        y_train: np.ndarray,
        y_test:  np.ndarray,
    ) -> tuple:
        """
        Run one addition step on a single mask (without logging).
        Returns (new_mask, improved, score, accuracy, n_selected).
        Useful for unit-testing the operator in isolation.
        """
        pop_in  = [mask]
        pop_out, _ = self.operate(pop_in, X_train, X_test, y_train, y_test)
        new_mask = pop_out[0]
        score, acc, n_sel, _ = self.fitness_fn.evaluate(
            new_mask, X_train, X_test, y_train, y_test
        )
        improved = int(np.sum(new_mask)) > int(np.sum(mask))
        return new_mask, improved, score, acc, n_sel

print("✅ AdditionOperator class defined.")

Population Factory

In [ ]:
def create_initial_population(
    n_features:     int,
    population_size: int  = 20,
    init_density:   float = 0.3,
    bias_computer:  FeatureBiasComputer = None,
    top_k_seed:     int   = 5,
    random_state:   int   = 42,
) -> list:
    """
    Create an initial population of binary masks for the parallel framework.

    Strategy
    --------
    - Most masks are random with `init_density` fraction of features active.
    - If a FeatureBiasComputer is provided, `top_k_seed` masks are seeded
      with the top-k highest-IG features already active (warm start).

    Parameters
    ----------
    n_features      : total number of features in the dataset
    population_size : number of masks to generate
    init_density    : fraction of features active in random masks
    bias_computer   : optional — if given, some masks are seeded with top-IG features
    top_k_seed      : how many top-IG masks to inject (only if bias_computer given)
    random_state    : reproducibility seed

    Returns
    -------
    population : list of np.ndarray binary masks
    """
    rng        = np.random.default_rng(random_state)
    population = []

    # ── Warm-start masks (IG-seeded) ──────────────────────────
    if bias_computer is not None and bias_computer.ig_scores_ is not None:
        n_warm = min(top_k_seed, population_size)
        for k in range(1, n_warm + 1):
            mask = np.zeros(n_features, dtype=int)
            top_indices = np.argsort(bias_computer.ig_scores_)[::-1][:k * max(1, n_features // 20)]
            mask[top_indices] = 1
            population.append(mask)
    else:
        n_warm = 0

    # ── Random masks ──────────────────────────────────────────
    n_random = population_size - len(population)
    for _ in range(n_random):
        mask = (rng.random(n_features) < init_density).astype(int)
        if mask.sum() == 0:              # ensure at least one feature active
            mask[rng.integers(n_features)] = 1
        population.append(mask)

    print(f"✅ Population created: {len(population)} masks "
          f"({n_warm} IG-seeded + {n_random} random, density≈{init_density:.0%})")
    return population

print("✅ create_initial_population() defined.")

Load Datasets

In [ ]:
# ─────────────────────────────────────────────
# Load all three datasets (same as Milestone 1)
# ─────────────────────────────────────────────
datasets = {}

for ds_name, sample_size in [("mnist", 5000), ("madelon", None), ("cifar10_gray", 5000)]:
    loader = DatasetLoader(ds_name)
    loader.load().preprocess(sample_size=sample_size)
    X_train, X_test, y_train, y_test = loader.split()
    datasets[ds_name] = {
        "X_train": X_train, "X_test": X_test,
        "y_train": y_train, "y_test": y_test,
        "n_features": X_train.shape[1]
    }

print("\n✅ All datasets loaded and split.")

Fit Information-Gain Bias

In [ ]:
# ─────────────────────────────────────────────
# Fit FeatureBiasComputer on training data
# (done once per dataset — reused for all generations)
# ─────────────────────────────────────────────
bias_computers = {}

for ds_name, data in datasets.items():
    print(f"\n🔍 Dataset: {ds_name.upper()}")
    bc = FeatureBiasComputer(bias_temperature=2.0, random_state=42)
    bc.fit(data["X_train"], data["y_train"])
    bias_computers[ds_name] = bc

print("\n✅ All bias computers fitted.")

Addition Operator Demo

In [ ]:
def run_addition_operator_demo(
    dataset_name:    str,
    data:            dict,
    bias_computer:   FeatureBiasComputer,
    population_size: int  = 20,
    n_generations:   int  = 10,
    n_additions:     int  = 3,
    use_bias:        bool = True,
) -> dict:
    """
    Runs the Addition Operator for n_generations on one dataset.

    Returns a dict with:
        - per-generation stats (score, improvement rate, avg features)
        - the final population
        - the ResultsLogger
    """
    print(f"\n{'='*60}")
    print(f"  ADDITION OPERATOR DEMO — {dataset_name.upper()}")
    mode_str = 'IG-biased' if use_bias else 'uniform (ablation)'
    print(f"  Pop: {population_size} | Generations: {n_generations} "
          f"| Additions/step: {n_additions} | Mode: {mode_str}")
    print(f"{'='*60}")

    X_train = data["X_train"]; X_test  = data["X_test"]
    y_train = data["y_train"]; y_test  = data["y_test"]
    n_features = data["n_features"]

    # ── Setup ─────────────────────────────────────────────────
    evaluator  = NaiveBayesEvaluator()
    fitness_fn = FitnessFunction(evaluator, alpha=0.01)
    logger     = ResultsLogger(dataset_name)

    operator = AdditionOperator(
        fitness_fn    = fitness_fn,
        bias_computer = bias_computer,
        n_additions   = n_additions,
        use_bias      = use_bias,
        random_state  = 42,
    )

    # ── Initial population ────────────────────────────────────
    population = create_initial_population(
        n_features      = n_features,
        population_size = population_size,
        init_density    = 0.2,
        bias_computer   = bias_computer if use_bias else None,
        top_k_seed      = 5,
        random_state    = 42,
    )

    gen_stats = []
    total_start = time.time()

    # ── Generation loop ───────────────────────────────────────
    for gen in range(1, n_generations + 1):
        population, stats = operator.operate(
            population, X_train, X_test, y_train, y_test,
            logger=logger, generation=gen
        )
        gen_stats.append(stats)

        # Compute avg active features in population
        avg_feats = np.mean([mask.sum() for mask in population])

        print(f"  Gen {gen:>2} | "
              f"Best Score: {stats['best_score']:.4f} | "
              f"Avg Score: {stats['avg_score']:.4f} | "
              f"Improved: {stats['n_improved']:>2}/{stats['population_size']} | "
              f"Avg Features: {avg_feats:.0f}/{n_features}")

    total_elapsed = time.time() - total_start
    logger.print_summary()
    print(f"\n  ⏱  Total time: {total_elapsed:.1f}s")

    return {
        "dataset_name" : dataset_name,
        "gen_stats"    : gen_stats,
        "final_population": population,
        "logger"       : logger,
        "use_bias"     : use_bias,
        "elapsed"      : total_elapsed,
    }


# ── Run on all 3 datasets ─────────────────────────────────────
addition_results = {}

for ds_name, data in datasets.items():
    addition_results[ds_name] = run_addition_operator_demo(
        dataset_name    = ds_name,
        data            = data,
        bias_computer   = bias_computers[ds_name],
        population_size = 20,
        n_generations   = 10,
        n_additions     = 5,
        use_bias        = True,
    )

print("\n\n🎉 Addition Operator demo complete on all datasets!")

Ablation: Biased vs Uniform Addition


In [ ]:
# Biased run already in addition_results["madelon"]
# Run uniform (ablation) separately
ablation_result = run_addition_operator_demo(
    dataset_name    = "madelon",
    data            = datasets["madelon"],
    bias_computer   = bias_computers["madelon"],
    population_size = 20,
    n_generations   = 10,
    n_additions     = 5,
    use_bias        = False,      # <── uniform selection
)

# ── Compare ───────────────────────────────────────────────────
biased_best  = max(s['best_score'] for s in addition_results['madelon']['gen_stats'])
uniform_best = max(s['best_score'] for s in ablation_result['gen_stats'])

print(f"\n📊 Madelon — Biased vs Uniform Ablation")
print(f"   IG-biased addition  best score : {biased_best:.4f}")
print(f"   Uniform  addition   best score : {uniform_best:.4f}")
delta = biased_best - uniform_best
sign  = '+' if delta >= 0 else ''
print(f"   Δ (biased - uniform)           : {sign}{delta:.4f}")

Export Results (for Hasan & Ayesha)

In [ ]:
import os

EXPORT_DIR = '/content/'
os.makedirs(EXPORT_DIR, exist_ok=True)

# ── CSV logs per dataset ──────────────────────────────────────
for ds_name, res in addition_results.items():
    res['logger'].save_csv(f"{EXPORT_DIR}addition_operator_log_{ds_name}.csv")

# ── Combined log ──────────────────────────────────────────────
combined = pd.concat(
    [res['logger'].to_dataframe() for res in addition_results.values()],
    ignore_index=True
)
combined.to_csv(f"{EXPORT_DIR}addition_operator_full_log.csv", index=False)
print(f"\n📄 Combined log: {len(combined)} rows → addition_operator_full_log.csv")

# ── Generation stats CSV ──────────────────────────────────────
gen_rows = []
for ds_name, res in addition_results.items():
    for s in res['gen_stats']:
        row = {"dataset": ds_name}
        row.update(s)
        gen_rows.append(row)

gen_df = pd.DataFrame(gen_rows)
gen_df.to_csv(f"{EXPORT_DIR}addition_operator_generation_stats.csv", index=False)
print(f"📄 Generation stats: {len(gen_df)} rows → addition_operator_generation_stats.csv")

# ── Final population masks ────────────────────────────────────
# Saved as a numpy .npz for easy loading by Hasan's coordinator
pop_data = {}
for ds_name, res in addition_results.items():
    pop_stack = np.stack(res['final_population'])  # shape: (pop_size, n_features)
    pop_data[ds_name] = pop_stack

np.savez(f"{EXPORT_DIR}addition_operator_final_populations.npz", **pop_data)
print("📄 Final populations saved → addition_operator_final_populations.npz")

# ── Summary table ─────────────────────────────────────────────
summary_rows = []
for ds_name, res in addition_results.items():
    stats      = res['gen_stats']
    best_score = max(s['best_score'] for s in stats)
    best_gen   = next(s['generation'] for s in reversed(stats)
                      if s['best_score'] == best_score)
    final_pop  = res['final_population']
    avg_feats  = np.mean([m.sum() for m in final_pop])
    n_total    = datasets[ds_name]['n_features']

    summary_rows.append({
        'dataset'            : ds_name,
        'n_features'         : n_total,
        'population_size'    : len(final_pop),
        'n_generations'      : len(stats),
        'best_score'         : round(best_score, 6),
        'best_score_gen'     : best_gen,
        'avg_final_features' : round(avg_feats, 1),
        'feature_%_used'     : round(100 * avg_feats / n_total, 1),
        'total_time_s'       : round(res['elapsed'], 2),
        'author'             : 'Aqib Ali',
        'milestone'          : 'M2-Member1',
    })

summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv(f"{EXPORT_DIR}addition_operator_summary.csv", index=False)

print("\n" + "="*65)
print("  MILESTONE 2 MEMBER 1 — ADDITION OPERATOR COMPLETE")
print("="*65)
print(summary_df.to_string(index=False))
print("\n✅ All files saved to /content/")